# AgroVisión — entrenamiento del modelo de café

De cero a artefacto firmado en unos 40 minutos con la GPU gratuita de Colab.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.
Sin GPU esto tarda horas.

---

### Qué produce

Cinco archivos indivisibles que la app instala como una unidad:

| archivo | qué es |
|---|---|
| `model.tflite` | el modelo cuantizado, ~5 MB, con **dos salidas**: logits y embedding |
| `labels.json` | orden canónico de las clases y su vínculo con el catálogo |
| `calibration.json` | temperatura, umbrales de las tres compuertas, centroides y matriz de precisión |
| `metrics.json` | F1 por clase, matriz de confusión, paridad, trazabilidad del dataset |
| `signature.bin` | firma Ed25519 sobre el conjunto — sin ella ningún teléfono lo instala |

### Datos

JMuBEN + JMuBEN2 · 58 555 imágenes de café arábica tomadas en campo en
Kirinyaga, Kenia, con acompañamiento de un patólogo · CC BY 4.0.

> Jepkoech, J.; Kenduiywo, B.; Mugo, D.; Chebet, E. (2021). *Arabica coffee leaf
> images dataset for coffee leaf disease detection and classification*.
> Data in Brief 36, 107142.

## 1 · Comprobar la GPU

Si esta celda no muestra una GPU, cambia el entorno de ejecución antes de seguir.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'SIN GPU — cambia el entorno de ejecución'

## 2 · Traer el pipeline

Se clona el repositorio. Si lo tienes privado, Colab te pedirá autenticación:
en ese caso usa un *personal access token* en la URL, o descomenta el bloque
de subida manual de más abajo.


In [ ]:
import pathlib, shutil

REPO = 'https://github.com/diegokld30/coffeApp-ml.git'   # ← ajusta si le pusiste otro nombre

destino = pathlib.Path('/content/agrovision-ml')
shutil.rmtree(destino, ignore_errors=True)
!git clone --depth 1 {REPO} {destino}

RAIZ = destino

# ── Alternativa sin git: subir el zip a mano ──
# from google.colab import files
# import zipfile
# subidos = files.upload()          # elige ml.zip
# with zipfile.ZipFile(next(iter(subidos))) as zf:
#     zf.extractall('/content/agrovision-ml')
# RAIZ = pathlib.Path('/content/agrovision-ml')

assert (RAIZ / 'pyproject.toml').exists(), f'No encuentro pyproject.toml en {RAIZ}'
print('paquete en', RAIZ)


## 3 · Instalar

Colab trae TensorFlow 2.20, pero el pipeline lo acota a `<2.20` para que el
`.tflite` que produzca lo pueda abrir el runtime **LiteRT 1.0.1** que lleva la
app. Un converter más nuevo que el runtime puede emitir versiones de operaciones
que el teléfono no conoce, y ese fallo aparece en campo, no aquí.

> Por eso pip baja TensorFlow y se queja de `ydf-tf`, `tensorflow-text` y
> `tf-keras`. **Es esperado:** el pipeline no importa ninguno de los tres. Solo
> usa `tensorflow`, `numpy`, `cryptography`, `rich`, `typer`, `yaml`, `requests`
> y `tqdm`.


In [ ]:
# Se filtran las quejas de pip sobre paquetes que este pipeline no importa.
%pip install -q -e {RAIZ} 2>&1 | grep -viE 'dependency resolver|requires tensorflow|is incompatible' | tail -3

import tensorflow as tf
from tensorflow import keras

dispositivos = [d.device_type for d in tf.config.list_physical_devices()]
print('tensorflow', tf.__version__, '· keras', keras.__version__)
print('dispositivos', dispositivos)

# Barrera dura: sin esto, «Ejecutar todas» seguiría en CPU y te enterarías tres
# horas después, no ahora.
assert 'GPU' in dispositivos, (
    'NO HAY GPU. Entorno de ejecución → Cambiar tipo de entorno → GPU T4. '
    'En CPU esto pasa de 40 minutos a varias horas.'
)

# Keras 3 es lo que espera el modelo de doble salida. Si alguien define
# TF_USE_LEGACY_KERAS, `from tensorflow import keras` devolvería Keras 2 y el
# grafo exportado podría diferir sin ningún error visible.
assert keras.__version__.startswith('3'), f'Se esperaba Keras 3 y hay {keras.__version__}'

print('✓ listo para entrenar')


## 4 · Drive: clave de firma y caché del dataset

Todo lo que vive en `/content` **muere con la sesión**. Esta celda pone en tu
Drive las dos cosas que no puedes permitirte perder:

- **La clave privada de firma.** Si la pierdes, el próximo modelo que firmes con
  otra clave será rechazado por todos los teléfonos que ya tengan la anterior
  embebida en el APK — y eso obliga a publicar una versión nueva de la app.
- **Los ZIP del dataset.** 1,75 GB que no querrás volver a descargar cada vez
  que se te caiga la sesión.

> ⚠️ Guardar la clave privada en Drive es lo pragmático, no lo más seguro.
> Descárgala también a un gestor de contraseñas. Quien acceda a tu Drive puede
> firmar modelos que la app instalará como legítimos (§15.6).


In [ ]:
import pathlib
from google.colab import drive

drive.mount('/content/drive')

BASE = pathlib.Path('/content/drive/MyDrive/agrovision')
SECRETOS = BASE / 'secretos'
SECRETOS.mkdir(parents=True, exist_ok=True)
CLAVE = SECRETOS / 'model_signing_private.pem'

if CLAVE.exists():
    print('ya hay una clave — se reutiliza')
else:
    !agrovision-ml keys --out {SECRETOS}

# Copia esta línea a mobile-android/gradle.properties si aún no está.
print((SECRETOS / 'model_signing_public.b64').read_text())

# ── Caché del dataset ──
DATOS = pathlib.Path('/content/datos')
DATOS.mkdir(exist_ok=True)
CACHE = BASE / 'raw'
CACHE.mkdir(parents=True, exist_ok=True)

# Solo los ZIP van a Drive. Las 58.555 imágenes descomprimidas se quedan en
# disco local: leerlas desde Drive sería lentísimo — cada archivo pasa por red
# y el entrenamiento se arrastraría.
enlace = DATOS / 'raw'
if not enlace.exists():
    enlace.symlink_to(CACHE)

print('caché en', CACHE)
!ls -la /content/datos/ | grep raw


## 5 · Descargar el dataset

~1,75 GB desde Mendeley, reanudable. Como los ZIP quedan en Drive, **solo se
descargan la primera vez**: en las siguientes corridas esta celda los encuentra
y pasa directo a descomprimir.


In [ ]:
# DATOS ya viene de la celda anterior, con raw/ enlazado a Drive.
!agrovision-ml download --config {RAIZ}/configs/coffee_v1.yaml --data {DATOS}


## 6 · Entrenar

Unos 30–40 minutos en una T4. El comando hace todo el recorrido:

```
reparto → etapa 1 (cabeza) → etapa 2 (ajuste fino) → calibración →
evaluación → exportación INT8 → paridad → aceptación → firma
```

Si el modelo no alcanza los mínimos de la receta, **se detiene antes de firmar**.

In [ ]:
ARTEFACTOS = pathlib.Path('/content/artefactos')

!agrovision-ml train \
    --config {RAIZ}/configs/coffee_v1.yaml \
    --data {DATOS} \
    --out {ARTEFACTOS} \
    --key {CLAVE}

## 7 · Revisar el informe

Lo que de verdad hay que mirar antes de publicar nada.

In [ ]:
import json

version = next(ARTEFACTOS.glob('v*'))
informe = json.loads((version / 'metrics.json').read_text())

print(f"F1 macro    {informe['evaluation']['macro_f1']}")
print(f"exactitud   {informe['evaluation']['accuracy']}")
print(f"temperatura {informe['calibration']['temperature']}")
print(f"ECE         {informe['calibration']['ece_before']} → {informe['calibration']['ece_after']}")
print(f"tamaño      {informe['export']['size_bytes'] / 1048576:.2f} MB")
print()
print('Por clase:')
for fila in informe['evaluation']['per_class']:
    print(f"  {fila['class_id']:30} F1 {fila['f1']:.4f}  ({fila['support']:,} imágenes)")
print()
print('De cada 100 fotos, la app respondería:')
for clave, valor in informe['evaluation']['gate_distribution'].items():
    print(f"  {clave:16} {valor * 100:5.1f}")

### Matriz de confusión

Dónde se confunde el modelo importa más que cuánto. Roya con mancha de hierro es
un error comprensible —ambas son manchas foliares— y recuperable por el
agrónomo. Confundir una hoja sana con una enferma es peor: dispara una
aplicación de fungicida que no hacía falta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

matriz = np.array(informe['evaluation']['confusion_matrix'])
nombres = [c['class_id'] for c in informe['evaluation']['per_class']]
normalizada = matriz / np.maximum(matriz.sum(axis=1, keepdims=True), 1)

figura, eje = plt.subplots(figsize=(7, 6))
eje.imshow(normalizada, cmap='YlOrBr', vmin=0, vmax=1)
eje.set_xticks(range(len(nombres)), nombres, rotation=45, ha='right')
eje.set_yticks(range(len(nombres)), nombres)
eje.set_xlabel('predicho'); eje.set_ylabel('real')
for i in range(len(nombres)):
    for j in range(len(nombres)):
        eje.text(j, i, f'{normalizada[i, j]:.2f}', ha='center', va='center',
                 color='white' if normalizada[i, j] > 0.5 else 'black', fontsize=9)
plt.tight_layout(); plt.show()

## 8 · Descargar los artefactos

Se copian a `mobile-android/app/src/main/assets/model/` para que viajen dentro
del APK como modelo de fábrica, y se publican por OTA desde la consola.

In [ ]:
import shutil

# La carpeta `trabajo/` son pesos intermedios de varios cientos de MB: no viaja.
shutil.rmtree(version / 'trabajo', ignore_errors=True)

paquete = shutil.make_archive('/content/agrovision-modelo', 'zip', version)
print(f'{paquete} · {pathlib.Path(paquete).stat().st_size / 1048576:.1f} MB')
files.download(paquete)

# ⚠️ La clave ya está en tu Drive, pero descárgala TAMBIÉN a un gestor de
# contraseñas: si pierdes el acceso al Drive, pierdes la capacidad de firmar
# y ningún teléfono aceptará la próxima versión del modelo.
files.download(str(CLAVE))

---

## Reentrenar cuando lleguen fotos del piloto

Este es el punto de todo el montaje. Para añadir imágenes propias:

1. Colócalas en `datos/images/<carpeta_de_la_clase>/` — las mismas carpetas que
   ya existen.
2. Sube `version` en `coffee_v1.yaml`.
3. Vuelve a ejecutar la celda 6.

El reparto entrenamiento/validación/prueba **no se recalcula al azar**: se deriva
del hash del nombre de cada archivo. Una foto que estaba en prueba sigue en
prueba aunque el dataset crezca, así que las métricas de la versión nueva son
comparables con las de la anterior. Con un reparto aleatorio no lo serían, y
nadie se daría cuenta.

### Para añadir una clase nueva

Añade su entrada a `classes:` en la receta **al final de la lista**. Insertarla
en medio recorre los índices de los logits y el modelo diría «roya» donde antes
decía «minador».

Cambiar el conjunto de clases es un **cambio mayor** de versión (§17.2) y exige
subir `minAppVersion`.

### Sobre el AUROC

Sin la opción `--ood`, el AUROC queda en `null` y el modelo puede ir a `draft` o
`internal`, **no a `production`**. Para medirlo hace falta una carpeta con fotos
que no sean ninguna de las cinco clases —otras plantas, suelo, manos, cielo—.
Las mejores llegarán solas: son justo las que la app manda a revisión con «no la
reconozco».